# Data Ingestion Pipeline

This notebook handles the initial data ingestion from various sources into the MS Fabric lakehouse.

## Execution Steps:
1. Connect to data sources (Azure SQL, Blob Storage, etc.)
2. Extract raw data
3. Perform initial data validation
4. Store raw data in Bronze layer

In [ ]:
# Import required libraries for MS Fabric
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import json

# MS Fabric specific imports
# from fabric import FabricDataFrame
# from fabric.lakehouse import Lakehouse

In [ ]:
# Configuration
LAKEHOUSE_NAME = "fabric_sandbox_lakehouse"
SOURCE_SYSTEMS = {
    "azure_sql": "server=example.database.windows.net;database=source_db",
    "blob_storage": "https://storageaccount.blob.core.windows.net/container",
    "api_endpoint": "https://api.example.com/data"
}

print(f"Starting data ingestion at {datetime.now()}")
print(f"Target lakehouse: {LAKEHOUSE_NAME}")

In [ ]:
# Function to ingest data from Azure SQL Database
def ingest_from_sql(connection_string, table_name):
    """
    Ingest data from Azure SQL Database
    
    Args:
        connection_string (str): SQL connection string
        table_name (str): Source table name
    
    Returns:
        pd.DataFrame: Ingested data
    """
    query = f"SELECT * FROM {table_name} WHERE created_date >= DATEADD(day, -1, GETDATE())"
    
    # Simulate data ingestion
    print(f"Ingesting from SQL table: {table_name}")
    
    # In real implementation, would use:
    # df = pd.read_sql(query, connection_string)
    
    # Sample data for demonstration
    sample_data = {
        'id': range(1, 1001),
        'customer_id': [f'CUST_{i:04d}' for i in range(1, 1001)],
        'transaction_amount': [100.0 + (i * 0.5) for i in range(1, 1001)],
        'transaction_date': [datetime.now() for _ in range(1000)],
        'source_system': ['azure_sql'] * 1000
    }
    
    return pd.DataFrame(sample_data)

# Ingest customer transaction data
transactions_df = ingest_from_sql(SOURCE_SYSTEMS['azure_sql'], 'customer_transactions')
print(f"Ingested {len(transactions_df)} transaction records")
print(transactions_df.head())

In [ ]:
# Function to save data to Bronze layer in Lakehouse
def save_to_bronze(df, table_name, partition_column=None):
    """
    Save DataFrame to Bronze layer in Delta format
    
    Args:
        df (pd.DataFrame): Data to save
        table_name (str): Target table name
        partition_column (str): Column to partition by
    """
    # Add metadata columns
    df['_ingestion_timestamp'] = datetime.now()
    df['_source_file'] = f'{table_name}_batch'
    
    bronze_path = f"/lakehouse/default/Tables/bronze_{table_name}"
    
    print(f"Saving {len(df)} records to Bronze layer: {bronze_path}")
    
    # In real implementation, would use:
    # df.to_delta(bronze_path, mode='append', partition_cols=[partition_column] if partition_column else None)
    
    return bronze_path

# Save to Bronze layer
bronze_transactions_path = save_to_bronze(transactions_df, 'customer_transactions', 'transaction_date')
print(f"Data saved to Bronze layer: {bronze_transactions_path}")

In [ ]:
# Data quality validation
def validate_data_quality(df, table_name):
    """
    Perform basic data quality checks
    
    Args:
        df (pd.DataFrame): Data to validate
        table_name (str): Table name for reporting
    
    Returns:
        dict: Validation results
    """
    validation_results = {
        'table_name': table_name,
        'total_records': len(df),
        'null_counts': df.isnull().sum().to_dict(),
        'duplicate_records': df.duplicated().sum(),
        'validation_timestamp': datetime.now().isoformat()
    }
    
    print(f"Data Quality Report for {table_name}:")
    print(f"  Total records: {validation_results['total_records']}")
    print(f"  Duplicate records: {validation_results['duplicate_records']}")
    print(f"  Null counts: {validation_results['null_counts']}")
    
    return validation_results

# Validate ingested data
validation_report = validate_data_quality(transactions_df, 'customer_transactions')

# Log validation results
with open('/tmp/validation_report.json', 'w') as f:
    json.dump(validation_report, f, indent=2, default=str)

print("\nData ingestion completed successfully!")
print(f"Next step: Run notebook 02_data_transformation.ipynb")